# Atelier Prompt Engineering — AI Business Assistant 
Ce notebook regroupe l'ensemble des réponses à l'atelier, organisées partie par partie.
## Contexte

L’entreprise souhaite mettre en place **AI Business Assistant**, un assistant IA polyvalent pour exploiter des documents, analyser des données, faire du machine learning et produire des résultats structurés. L’objectif de l’atelier est de construire progressivement et d’améliorer les prompts permettant de réaliser ces différentes tâches.

# Partie 1 — Anatomie d’un prompt

### Problème
Analyser les retours de clients d’une entreprise.

**Décomposition en composantes** : rôle, contexte, tâche, contraintes/format.


**Rôle**: Tu es un analyste spécialisé dans l’analyse des retours clients.

**Contexte** : Une entreprise de e‑commerce reçoit chaque semaine un grand volume d’avis clients provenant de différentes sources — réseaux sociaux, courriels et formulaires de satisfaction. Ces retours, bien que riches en informations, ne font pas encore l’objet d’une analyse systématique.

**Tâche**: Analyse l’ensemble des avis clients fournis ci‑dessous et produis une synthèse structurée mettant en évidence les tendances, les points forts et les axes d’amélioration.

**Contraintes**: 
1.Indique pour chaque avis le sentiment général : positif, négatif ou neutre.

2.Regroupe les avis par sujet : livraison, produit, service client, prix, application.

3.Mets en avant les points positifs les plus cités et les problèmes les plus fréquents.

4.Résume les tendances principales observées dans les retours.

5.Propose deux ou trois actions concrètes pour améliorer l’expérience client.

6.Utilise uniquement les informations présentes dans les avis, sans ajout ni interprétation.

7.Rédige de façon claire, neutre et concise, sans jugement personnel.

**Format de sortie** :
1. Points positifs
2. Problèmes principaux
3. Tendances
4. Recommandations

----
## Partie 2 — Comparer les techniques de prompting

**Objectif** : tester et comparer les résultats obtenus avec différentes approches de prompt (zero‑shot, one‑shot, few‑shot et prompt structuré).

**Tâche** : classer le commentaire suivant selon le sentiment exprimé :

#### <mark style= "background-color: lightblue">**Zero-shot** </mark>
📋**Prompt:**

Classe le commentaire suivant parmi : positif, négatif, neutre.
Commentaire : « Le service est rapide mais l'application plante régulièrement. »

💬**Réponse du LLM**: <mark> Neutre </mark>


**Evaluation**: Sans exemple ni règle de décision, le modèle traite les deux informations (rapidité / plantages) comme équivalentes en poids et choisit la catégorie "neutre" par défaut face à l'ambiguïté, faute d'indication sur comment trancher.



#### <mark style= "background-color: lightblue">**One-shot**</mark>
📋**Prompt:**

Exemple :

Commentaire : « Le livreur a été très aimable et ponctuel. »
Classe : positif

Maintenant, classe le commentaire suivant parmi positif, négatif ou neutre :
« Le service est rapide mais l'application plante régulièrement. »

💬**Réponse du LLM:**<mark>négatif</mark>

 **Evaluation:** L'exemple positif unique manque de nuance : il n'apprend pas au modèle à peser le pour et le contre d'un avis mixte, ce qui le pousse à privilégier la critique technique (« plante ») et à trancher arbitrairement vers le négatif.




#### <mark style= "background-color: lightblue">**Few-shot** </mark>
📋**Prompt** : 

Classe chaque texte en positif, négatif ou neutre.

Texte : « Le livreur a été très aimable et ponctuel. » → positif

Texte : « Ma commande n’est jamais arrivée. » → négatif

Texte : « Le produit correspond à la description, rien de plus. » → neutre

Texte : « Prix correct mais le service client ne répond jamais. » → négatif

Texte : « Le service est rapide mais l’application plante régulièrement. » → ?

💬**Réponse du LLM:** <mark>Neutre</mark>

**Evaluation:** Grâce aux multiples exemples, le modèle comprend mieux la tâche, mais il hésite toujours sur les avis partagés. Comme le prompt ne lui dit pas si un bug technique est plus grave qu'un service rapide, le modèle choisit neutre au hasard plutôt que négatif.

#### <mark style= "background-color: lightblue">**Prompt structuré** </mark>

📋**Prompt**

**Rôle**: Tu es un système d'analyse de sentiment client pour une entreprise de services numériques.

**Contexte**: Les commentaires clients peuvent contenir plusieurs aspects, parfois contradictoires (points positifs et négatifs dans le même commentaire).

**Tâche** : Classer le commentaire suivant dans une seule catégorie parmi : positif, négatif, neutre.

**Règles de décision** :
- Si le commentaire est globalement favorable sans défaut majeur → positif
- Si le commentaire mentionne un défaut qui impacte fortement l'usage (bug bloquant, panne, dysfonctionnement récurrent), même en présence d'un point positif → négatif
- Si les aspects positifs et négatifs sont équivalents en importance, sans dominance claire → neutre

**Format de sortie** : un seul mot parmi "positif", "négatif" ou "neutre", sans explication ni ponctuation.

**Commentaire à classer** : « Le service est rapide mais l'application plante régulièrement. »

--

💬**Réponse de LLM:** <mark>Négatif</mark>


**Évaluation (Prompt structuré)**

* **Points forts :** L'ajout de règles de décision explicites enlève toute ambiguïté sur la gestion des avis mixtes. Le modèle applique directement la règle stipulant qu'un dysfonctionnement récurrent ("plante régulièrement") implique une classification **négatif**.
* **Respect des contraintes :** Le format imposé (un seul mot, sans explication ni ponctuation) est parfaitement suivi, facilitant une exploitation automatique.
* **Bilan :** C'est la méthode la plus fiable et constante pour cadrer le comportement du modèle sur des tâches complexes.

---
## Partie 3 — Prompt Engineering et raisonnement

### 1. Décomposition du prompt
 « Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations » manque de structure. Voici son anatomie :
 
 -  Tâche principal (Instruction) : Analyser des avis clients, identifier des problèmes et suggérer des actions. 
 - Entrée (Données) : Absente (le texte d'avis clients à analyser n'est pas fourni). 
 - Rôle / Persona : Absent (pas -de rôle d'analyste expérience client spécifié).
 - Contraintes et Format : Absents (aucun format précisé comme JSON ou puces, pas de limite de mots, pas de définition de ce qu'est un problème "important").

### 2.1 Prompt amélioré
📋**Prompt**

Rôle : Tu es un analyste chargé d'exploiter des avis clients pour une entreprise de e-commerce.

Tâche : À partir des avis fournis ci-dessous, identifie les 3 problèmes les plus fréquemment mentionnés et propose une recommandation actionnable pour chacun.

Contraintes :
- Base-toi uniquement sur le contenu des avis fournis, sans aucune exception.
- N'invente aucune information, cause, chiffre ou détail absent des avis.
- Ne suppose aucune intention ou émotion du client qui ne soit pas explicitement exprimée.
- Si moins de 3 problèmes distincts sont identifiables, n'en liste que le nombre réellement présent — ne complète pas artificiellement.
- Chaque problème cité doit être accompagné d'une référence à l'avis correspondant (numéro de l'avis).
- Chaque recommandation doit être directement liée au problème identifié, sans généralité vague (ex : interdiction de recommandations du type "améliorer la qualité de service" sans précision).
- N'utilise aucune connaissance externe sur l'entreprise, le secteur ou des cas similaires.

Format de sortie : Liste numérotée. Pour chaque élément : Problème (avec n° d'avis) → Recommandation.

Avis clients :
1. « Livraison en retard de 3 jours, aucune excuse reçue. »
2. « Le produit est de bonne qualité mais l'emballage était endommagé. »
3. « Service client injoignable pendant plus d'une semaine. »
4. « Très satisfait, rien à signaler. »
5. « Le prix a augmenté sans prévenir, c'est frustrant. »

💬**Réponse de LLM:**

1. Problème (Avis 1 & 3) : Retards de livraison sans communication et service client totalement injoignable pendant plus d'une semaine. → Recommandation : Rétablir un canal de communication fonctionnel pour répondre aux clients et envoyer une notification automatique d'excuse lors des retards de livraison.

2. Problème (Avis 2) : Emballages des colis arrivant endommagés lors de la réception. → Recommandation : Renforcer la protection physique des emballages lors du conditionnement avant l'expédition.

3. Problème (Avis 5) : Augmentation des prix appliquée sans notification préalable aux clients. → Recommandation : Informer à l'avance les clients de toute hausse de tarif avant son application.


### 2.2 Vérification par un second prompt
📋**Prompt**

Examine ta réponse précédente à la tâche d'analyse d'avis clients avec la plus grande rigueur, en te basant sur les critères suivants :

1. Informations non justifiées : chaque problème cité est-il explicitement et textuellement présent dans l'avis référencé ? Signale toute reformulation qui dépasse le contenu littéral.
2. Contradictions : y a-t-il des incohérences entre les problèmes identifiés, ou entre un problème et sa recommandation ?
3. Informations absentes : un problème significatif mentionné dans les avis a-t-il été omis ?
4. Hallucinations : y a-t-il des détails, causes, chiffres ou intentions ajoutés qui ne figurent pas dans les avis ?
5. Respect des contraintes : le format (n° d'avis, recommandation spécifique, pas de connaissance externe) a-t-il été strictement respecté ?
6. Neutralité : la réponse a-t-elle évité tout jugement de valeur non fondé sur les avis (ex : qualifier un client de "difficile") ?

Pour chaque critère, indique explicitement : conforme / non conforme, avec justification. Si non conforme, corrige la réponse et indique précisément la modification apportée.

💬**Réponse de LLM:**

**1. Informations non justifiées : Non conforme**

* **Justification :** Le point 1 fusionne les avis 1 et 3 et ajoute « sans communication », alors que l'avis 1 parle uniquement de l'absence d'excuses.
* **Correction :** Séparer le retard de livraison sans excuses (Avis 1) du service client injoignable (Avis 3).

**2. Contradictions : Conforme**

* **Justification :** Chaque recommandation répond directement au problème associé sans incohérence.

**3. Informations absentes : Conforme**

* **Justification :** L'avis 4 (« Très satisfait ») ne contient pas de problème. Tous les avis négatifs (1, 2, 3 et 5) ont été traités.

**4. Hallucinations : Non conforme**

* **Justification :** L'expression « lors du conditionnement avant l'expédition » (point 2) ajoute un détail sur le processus logistique non mentionné dans l'avis.
* **Correction :** Supprimer cette précision et limiter la recommandation au renforcement des emballages.

**5. Respect des contraintes : Conforme**

* **Justification :** Les numéros d'avis, les recommandations ciblées et le format en liste numérotée ont été respectés.

**6. Neutralité : Conforme**

* **Justification :** Aucun jugement sur les clients ni interprétation d'émotions non formulées.


### **Réponse corrigée**

1. Problème (Avis 1) : Livraison en retard de 3 jours sans excuses reçues. → Recommandation : Envoyer un message d'excuse au client en cas de retard de livraison.
2. Problème (Avis 2) : Emballage du produit endommagé à l'arrivée. → Recommandation : Renforcer la protection des emballages utilisés.
3. Problème (Avis 3) : Service client injoignable pendant plus d'une semaine. → Recommandation : Rendre le service client accessible pour répondre aux sollicitations.



---
## Partie 4 — Sorties structurées
###  <mark style= "background-color: grey">**Prompt JSON** </mark>

 <mark style= "background-color: lightblue">**📋Prompt** </mark>


Analyse le commentaire du client et retourne uniquement un objet JSON valide.

Le JSON doit contenir exactement ces 5 champs :
- sentiment : chaîne de caractères, valeur parmi "positif", "negatif", "neutre"
- categorie : chaîne de caractères, par exemple "livraison", "produit", "service" ou "autre"
- urgence : chaîne de caractères, valeur parmi "faible", "moyenne", "élevée"
- probleme : chaîne de caractères décrivant brièvement le problème
- confiance : nombre décimal compris entre 0 et 1

Commentaire :
"Le client est mécontent car sa commande est arrivée avec trois jours de retard."

Ne retourne aucun texte en dehors du JSON.

 <mark style= "background-color: lightblue">💬**Réponse de LLM:** </mark>


{
  "sentiment": "negatif",

  "categorie": "livraison",

  "urgence": "moyenne",

  "probleme": "Retard de livraison de trois jours",
  
  "confiance": 0.95
}

#### <mark style= "background-color: grey">**Régles de validation** </mark>

1. La réponse doit être un JSON valide.
2. Elle doit contenir exactement les 5 propriétés demandées.
3. Aucune propriété supplémentaire n'est autorisée.
4. sentiment doit respecter les valeurs autorisées.
5. urgence doit respecter les valeurs autorisées.
6. confiance doit être comprise entre 0 et 1.


---
## Partie 5 — Prompts pour les applications métier

### 1. Résumer un document
 <mark style= "background-color: lightblue">**📋Prompt** </mark>

Rôle : Tu es un assistant chargé de résumer des documents professionnels.

Tâche : Résume le document fourni ci-dessous en respectant les contraintes suivantes :

Longueur maximale : 250 mots ;
Conserver uniquement les informations factuelles présentes dans le document ;
Identifier clairement les objectifs mentionnés ;
Identifier clairement les résultats mentionnés ;
Identifier clairement les recommandations mentionnées ;
N'inventer aucune information absente du document.

Format de réponse :
Objectifs :
Résultats :
Recommandations :
Résumé :

Document :
"""
DOCUMENT À ANALYSER
"""

 <mark style= "background-color: lightblue">💬**Réponse de LLM:** </mark>

**Objectifs :**

* Automatiser le résumé de documents, l'extraction d'informations, la classification des demandes et la traduction de contenus professionnels.


* Améliorer la rapidité du traitement des demandes internes.



**Résultats :**

* Un prototype a été testé sur 120 documents professionnels.


* Le temps moyen de première analyse par document est passé de 18 minutes à 6 minutes.


* Les utilisateurs ayant participé aux tests ont rapporté une meilleure organisation des informations.



**Recommandations :**

* Poursuivre les tests sur des types de documents plus variés.


* Ajouter une étape de validation pour les sorties structurées.


* Mettre en place une vérification humaine pour les informations importantes avant toute utilisation.



**Résumé :**
Dans le cadre du projet AI Business Assistant, l'entreprise cherche à réduire le temps consacré aux tâches répétitives et à faciliter l'accès aux données internes pour ses collaborateurs. Le projet vise à automatiser le traitement des documents (résumé, extraction, classification, traduction) et à accélérer le traitement des demandes internes. Les tests menés sur un prototype avec 120 documents ont permis de réduire le temps moyen d'analyse par document de 18 à 6 minutes, tout en apportant une meilleure organisation des informations aux utilisateurs. Pour la suite, il est préconisé d'étendre les tests à des documents plus variés, d'ajouter une validation des sorties structurées et d'instaurer une vérification humaine obligatoire pour les données critiques.

### 2. Prompt – Traduction français → anglais
<mark style= "background-color: lightblue">**📋Prompt** </mark>

**Rôle** : Tu es un traducteur professionnel spécialisé dans les documents techniques et métier.

**Tâche** : Traduis le document français fourni ci-dessous vers l'anglais, en respectant strictement les contraintes suivantes :

Conserver intégralement le sens du texte source ;
Conserver la structure du document (paragraphes, listes, titres, ordre des sections) ;
Conserver les termes techniques dans leur forme appropriée en anglais, sans les traduire de façon approximative ;
Ne pas résumer le contenu ; traduire l'intégralité du texte ;
Ne rajouter aucune information absente du texte source.

Document à traduire :

"""
DOCUMENT À TRADUIRE
"""

 <mark style= "background-color: lightblue">💬**Réponse de LLM:** </mark>


**User Guide – AI Business Assistant**

**Presentation**

AI Business Assistant helps employees analyze documents, extract information, and prepare professional responses.
**How It Works**
The user provides a document or a request. The system analyzes the available information and produces a response tailored to the requested processing.
**Security**
Sensitive information must be verified before being transmitted to the system. The results generated by the assistant must be checked when they are used to make an important decision.

### 3. Prompt – Classification d'un ticket informatique

<mark style= "background-color: lightblue">**📋Prompt** </mark>

Tu es un assistant chargé de classifier les tickets informatiques.

Analyse le ticket fourni et attribue-lui une seule catégorie parmi :

- réseau
- logiciel
- matériel
- sécurité
- accès
- autre

Choisis la catégorie qui correspond le mieux au problème décrit.

Ajoute une justification courte basée uniquement sur les informations présentes dans le ticket.

Retourne uniquement un JSON valide avec exactement les deux champs suivants :
{
  "categorie": "...",
  "justification": "..."
}

Ticket :
"""
[TICKET A INSERER]
"""

 <mark style= "background-color: lightblue">💬**Réponse de LLM:** </mark>
 
{
 **"categorie"**: "réseau",
 
 **"justification"**: "Le problème concerne l'impossibilité d'accéder au serveur de fichiers via la connexion au réseau Wi-Fi du bureau[cite: 3]."
  
}
